# 1.1 · PD training

How every configuration behaved *while it trained* — loss, real-data AUC, all logged metrics, hardware and gradient flow. Reads the per-arm progress and telemetry manifests, so it works on a partial sweep.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import training_plots, figures, style

style.apply()   # ONE shared style: identical colours in every notebook
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "pd"
# Every started arm leaves output/manifests/<run>__progress.csv and __telemetry.csv; these read
# them, so a partial sweep still plots. Clears THIS notebook's figure folder before drawing.
FIGS = figures.FigureSaver("1.1_pd_training")

## 1. Training loss

In [ ]:
FIGS.save(training_plots.training_loss(TASK), "training_loss",
    caption="Training loss against optimisation step for every PD arm (grey), the mean across arms (black), and the best and worst arms by final real-data AUC (highlighted).");

## 2. Real-data AUC over training

In [ ]:
FIGS.save(training_plots.metric_over_training(TASK), "metric_over_training",
    caption="Real-data AUC, averaged over the evaluation datasets, against training step; one line per arm coloured by prior (credit versus control), with the across-arm mean in black.");

## 3. Every logged evaluation metric

In [ ]:
for _p in range(1, training_plots.metric_pages(TASK) + 1):
    FIGS.save(training_plots.all_eval_metrics(TASK, _p), f"eval_metrics_p{_p}",
        caption="Each logged real-data evaluation metric, averaged over arms and datasets, against training step; the arrow in each panel title marks the improving direction.");

## 4. Per-configuration training curves

In [ ]:
for _p in range(1, training_plots.config_pages(TASK) + 1):
    FIGS.save(training_plots.per_config(TASK, _p), f"per_config_p{_p}",
        caption="Per-arm training curves against step: train loss (grey, left axis) and real-data AUC (blue, right axis), one panel per configuration.");

## 5. Best versus worst configuration

In [ ]:
FIGS.save(training_plots.best_and_worst(TASK), "best_and_worst",
    caption="Train loss and real-data AUC against training step for the best and worst arm by final AUC.");

## 6. Hardware during training

In [ ]:
FIGS.save(training_plots.hardware(TASK), "hardware",
    caption="GPU utilisation, throughput and peak allocated memory against training step, pooled across arms; the dashed line marks 70 percent utilisation.");

## 7. Per-block gradient flow

In [ ]:
FIGS.save(training_plots.gradient_flow(TASK), "gradient_flow",
    caption="Mean per-block gradient L2 norm (column encoder, row encoder, ICL blocks, head) against training step, on a logarithmic axis.");

## Summary

In [ ]:
print(training_plots.training_summary(TASK))
print()
print(FIGS.summary())